In [1]:
import cv2
from ultralytics import YOLO

# Load model YOLO11S
model = YOLO("yolo11s.pt") 

# URL CCTV 
cctv_url = "https://cctvjss.jogjakota.go.id/atcs/ATCS_Simpang_Demangan_View_Selatan.stream/chunklist_w735060430.m3u8"
cap = cv2.VideoCapture(cctv_url)

if not cap.isOpened():
    print("Gagal membuka stream CCTV.")
    exit()

print("Sistem Deteksi Semua Kendaraan Aktif. Tekan 'q' untuk keluar.")

while True:
    ret, frame = cap.read()
    if not ret:
        continue

    # Deteksi semua kelas kendaraan dengan conf=0.18
    results = model(frame, classes=[2, 3, 5, 7], conf=0.18, iou=0.45, verbose=False)

    # Inisialisasi hitungan per jenis kendaraan
    hitung_mobil = 0
    hitung_motor = 0
    hitung_bus = 0
    hitung_truk = 0

    # Ambil frame hasil deteksi awal bawaan YOLO
    annotated_frame = results[0].plot()

    # Lakukan pengecekan logika filter ukuran
    for box in results[0].boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        
        # Hitung dimensi kotak
        box_width = x2 - x1
        box_height = y2 - y1
        box_area = box_width * box_height
        
        # Ambil ID kelas ke dalam variabel biasa
        class_id = int(box.cls[0])

        # --- LOGIKA FILTER AMAN ---
        if class_id in [2, 5, 7] and (box_area < 2000 or box_width < 45):
            class_id = 3 # Dianggap Motor untuk perhitungan statistik

        # Hitung jumlah berdasarkan class_id yang sudah difilter
        if class_id == 2:
            hitung_mobil += 1
        elif class_id == 3:
            hitung_motor += 1
        elif class_id == 5:
            hitung_bus += 1
        elif class_id == 7:
            hitung_truk += 1

    # Total semua kendaraan setelah difilter
    total_kendaraan = hitung_mobil + hitung_motor + hitung_bus + hitung_truk

    # Tentukan Status Kemacetan
    status = "Lancar"
    if total_kendaraan > 18:  
        status = "Macet Total"
    elif total_kendaraan > 10:
        status = "Padat Merayap"
    elif total_kendaraan > 5:
        status = "Ramai Lancar"

    # HASIL VISUALISASI 
    cv2.putText(annotated_frame, f"Total Kendaraan: {total_kendaraan} ({status})", (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)
    cv2.putText(annotated_frame, f"Motor: {hitung_motor} | Mobil: {hitung_mobil}", (20, 95), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
    cv2.putText(annotated_frame, f"Bus: {hitung_bus} | Truk: {hitung_truk}", (20, 125), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
    
    # 🌟 SOLUSI: Perkecil resolusi tampilan window (misal jadi HD 720p atau sesuai kebutuhan)
    # Ini menjamin seluruh area video masuk ke monitor tanpa terpotong/efek zoom
    frame_tampilan = cv2.resize(annotated_frame, (854, 480)) # Ukuran 480p atau (1280, 720) untuk 720p
    
    # Tampilkan frame yang sudah di-resize
    cv2.imshow("Hackathon - Deteksi Semua Jenis Kendaraan", frame_tampilan)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Sistem Deteksi Semua Kendaraan Aktif. Tekan 'q' untuk keluar.
